In [0]:
# Databricks notebook source
from pyspark.sql import functions as F
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "churn_dev")
catalog, schema = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")

bronze = spark.table(f"{catalog}.{schema}.bronze_customers")

median_charge = bronze.approxQuantile("monthly_charges", [0.5], 0.01)[0]

silver = (bronze
    .dropDuplicates(["customer_id"])
    .fillna({"monthly_charges": median_charge})
    .filter((F.col("tenure_months") > 0) & (F.col("monthly_charges") > 0)))

# Data-quality gate: fail the task (and the job) if the data is bad
assert silver.count() > 10000, "Silver row count unexpectedly low"
assert silver.filter(F.col("customer_id").isNull()).count() == 0, "Null customer_id"

silver.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_customers")